# CouchDB: load, verify and query the Grab Mobility & Fulfillment database

This notebook runs the CouchDB implementation end to end (report sections 2.3, 3.1 and 3.2):

1. connect to CouchDB and show the environment,
2. *(optional)* load the canonical dataset into CouchDB,
3. verify that CouchDB holds exactly the canonical rows,
4. run each of the 25 essential operations once and check its result,
5. query the database freely (documents, views, Mango),
6. *(optional)* run the benchmark (1 warm-up + 3 measured runs per operation),
7. run the integrity-rule tests and write an audit manifest.

**Audit trail.** Every run writes its evidence to `benchmarks/results/couchdb_<dataset>/`:
the environment and code version (`environment.json`), the load and verification reports,
the result of every operation, the benchmark runs, the test output, and a `MANIFEST.json`
with the SHA-256 of each file. Save the notebook afterwards so its outputs are kept as well.

**Before running**

```bash
cp .env.example .env            # then set the passwords
docker compose --env-file .env -f docker/compose.yaml up -d --build
uv sync --frozen
```

Open this notebook with the project's `.venv` kernel and run the cells from top to bottom.

## Parameters

* `DATA_DIR`: the dataset to load and check against. `data/canonical` is the full dataset
  (2,000,000 orders). For a quick trial on real CouchDB, any folder with the same 13 CSV
  files works, e.g. a sample of every 50th order.
* `RUN_LOAD = True` **drops** the four databases of this dataset and loads `DATA_DIR` again. For
  the full dataset this takes a while (loading plus building the view indexes). The full data
  uses `grab_*`, any other dataset `grab_<name>_*`, so they never overwrite each other.
* When switching `DATA_DIR`, use **Restart Kernel and Run All**.
* `RUN_BENCHMARK = True` runs the benchmark runner over the 25 operations.
* `RUN_TESTS = True` runs the integrity-rule and unit tests (temporary databases).
* `PUBLISH = True` copies the evidence (not the large reference file) to
  `benchmarks/published/couchdb_<dataset>/`, which is tracked by Git.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / 'data' / 'canonical'
RUN_LOAD = False
RUN_BENCHMARK = False
RUN_TESTS = True
PUBLISH = False
PREVIEW_ROWS = 10

DATASET = DATA_DIR.name
# Each dataset lives in its own databases (grab_* for the full data, grab_sample_* for the
# sample), so loading one never wipes the other. Restart the kernel when switching datasets.
os.environ['COUCHDB_DB_PREFIX'] = 'grab' if DATASET == 'canonical' else f'grab_{DATASET}'
OUT_DIR = ROOT / 'benchmarks' / 'results' / f'couchdb_{DATASET}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_FILE = ROOT / 'data' / 'reference' / f'couchdb_reference_{DATASET}.json'
print(f"dataset: {DATA_DIR}  ->  databases {os.environ['COUCHDB_DB_PREFIX']}_*")
print(f'evidence: {OUT_DIR}')

# Keep the Mac awake while this kernel runs (long loads and benchmarks).
import shutil, subprocess
if shutil.which('caffeinate'):
    subprocess.Popen(['caffeinate', '-i', '-w', str(os.getpid())])

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

## 1. Connect and record the environment

In [ ]:
from couchdb import design, load, reference, setup_db, verify_load
from couchdb.client import CouchClient
from couchdb.queries import OPERATIONS, GrabCouch
from couchdb.settings import DATABASES, get_settings

settings = get_settings()
admin = CouchClient.admin(settings, timeout=None)
app = CouchClient.app(settings, timeout=None)
grab = GrabCouch(app, settings)
DB = {name: settings.db(name) for name in DATABASES}

server = setup_db.wait_until_up(admin)
print(f"CouchDB {server.get('version')} at {settings.url}")
print(f"JavaScript engine: {setup_db.javascript_engine(admin)}")
print(f"Operations run as: {app.user} (admin: {admin.user})")

In [ ]:
import hashlib
import importlib.metadata as metadata
import platform
from datetime import UTC, datetime

def sh(*command):
    """Output of a command, or a note when it is not available on this machine."""
    try:
        return subprocess.run(command, capture_output=True, text=True, timeout=30,
                              cwd=ROOT).stdout.strip()
    except Exception as error:
        return f'unavailable ({type(error).__name__})'

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifest_file = DATA_DIR / 'manifest.json'
environment = {
    'run_started_utc': datetime.now(UTC).isoformat(),
    'git_commit': sh('git', 'rev-parse', 'HEAD'),
    'git_branch': sh('git', 'rev-parse', '--abbrev-ref', 'HEAD'),
    'git_uncommitted_changes': sh('git', 'status', '--porcelain', '--untracked-files=no').splitlines(),
    'os': platform.platform(),
    'cpu': sh('sysctl', '-n', 'machdep.cpu.brand_string') if sys.platform == 'darwin' else platform.processor(),
    'cpu_cores': os.cpu_count(),
    'ram_gb': (round(int(sh('sysctl', '-n', 'hw.memsize')) / 2**30, 1)
               if sh('sysctl', '-n', 'hw.memsize').isdigit() else None),
    'python': platform.python_version(),
    'packages': {p: metadata.version(p) for p in ('requests', 'pandas', 'pytest')},
    'docker': sh('docker', 'version', '--format', '{{.Server.Version}}'),
    'docker_resources': sh('docker', 'info', '--format', 'cpus={{.NCPU}} memory_bytes={{.MemTotal}}'),
    'couchdb_version': server.get('version'),
    'couchdb_features': server.get('features'),
    'javascript_engine': setup_db.javascript_engine(admin),
    'couchdb_url': settings.url,
    'app_user': app.user,
    'dataset': str(DATA_DIR),
    'dataset_orders': (json.loads(manifest_file.read_text())['tables']['service_orders']['rows']
                       if manifest_file.is_file()
                       else sum(1 for _ in open(DATA_DIR / 'service_orders.csv')) - 1),
    'dataset_sha256': ({t: v['sha256'] for t, v in json.loads(manifest_file.read_text())['tables'].items()}
                       if manifest_file.is_file()
                       else {f.stem: sha256(f) for f in sorted(DATA_DIR.glob('*.csv'))}),
    'parameters': {'RUN_LOAD': RUN_LOAD, 'RUN_BENCHMARK': RUN_BENCHMARK, 'RUN_TESTS': RUN_TESTS},
}
(OUT_DIR / 'environment.json').write_text(json.dumps(environment, indent=2) + '\n')
if environment['git_uncommitted_changes']:
    print('WARNING: uncommitted code changes; results are not tied to one commit:',
          environment['git_uncommitted_changes'])
pd.Series({k: v for k, v in environment.items() if k != 'dataset_sha256'}).to_frame('value')

In [ ]:
def database_overview():
    rows = []
    for name, db in DB.items():
        if not admin.db_exists(db):
            rows.append({'database': db, 'exists': False})
            continue
        info = admin.db_info(db)
        sizes = info.get('sizes') or {}
        rows.append({'database': db, 'exists': True, 'documents': info['doc_count'],
                     'deleted': info['doc_del_count'],
                     'file_MB': round(sizes.get('file', 0) / 2**20, 1),
                     'data_MB': round(sizes.get('external', 0) / 2**20, 1),
                     'part2_tables': ', '.join(DATABASES[name])})
    return pd.DataFrame(rows)

database_overview()

## 2. Load the dataset (optional)

In [ ]:
LOAD_REPORT = OUT_DIR / 'load_report.json'
if RUN_LOAD:
    exit_code = load.main(['--data-dir', str(DATA_DIR), '--reset', '--report', str(LOAD_REPORT)])
    print('load exit code:', exit_code)

if LOAD_REPORT.is_file():
    report = json.loads(LOAD_REPORT.read_text())
    print(f"Loaded from {report['data_dir']} with CouchDB {report['couchdb_version']}, "
          f"batch {report['batch_size']}, {report['workers']} workers")
    display(pd.DataFrame([{'database': db, **v} for db, v in report['databases'].items()]))
    display(pd.DataFrame(report.get('index_builds', [])))
else:
    print('No load report yet: set RUN_LOAD = True to load DATA_DIR.')

## 3. Verify the load (section 3.1)

Every Part 2 table must have the same number of rows in CouchDB (counted by the
`entity_counts` views, e.g. trips embedded in order documents), and a random sample of
documents must turn back into exactly the original CSV rows.

In [ ]:
VERIFY_REPORT = OUT_DIR / 'verify_report.json'
verify_load.main(['--data-dir', str(DATA_DIR), '--sample', '1000', '--report', str(VERIFY_REPORT)])
verification = json.loads(VERIFY_REPORT.read_text())
display(pd.DataFrame(verification['tables']))
pd.DataFrame([{'entity': k, 'sampled': v['sampled'], 'mismatches': len(v['mismatches'])}
              for k, v in verification['samples'].items()])

## 4. Run the essential operations (section 3.2)

Expected results are computed straight from the CSV files of `DATA_DIR`
(`couchdb/reference.py`, which re-implements the Part 2 SQL), and every result is compared
with them. Write operations prepare their starting state, run, are checked by reading the
documents back, and are then undone, so the database ends as it started.

In [ ]:
os.environ['COUCHDB_REFERENCE_FILE'] = str(REFERENCE_FILE)
stale = (not REFERENCE_FILE.is_file()
         or json.loads(REFERENCE_FILE.read_text())['meta']['data_dir'] != str(DATA_DIR))
if stale:
    reference.main(['--data-dir', str(DATA_DIR), '--output', str(REFERENCE_FILE)])
else:
    print(f'Using expected results in {REFERENCE_FILE}')

In [ ]:
from benchmarks.adapters import couchdb as adapter

SCENARIOS = {s.operation_id: s for s in adapter.create_scenarios()}
from couchdb import client as couch_client_module
ORIGINAL_REQUEST, ORIGINAL_JSON = CouchClient.request, CouchClient.json
ORIGINAL_GET_DOC = CouchClient.get_doc
ORIGINAL_ITER_ROWS = couch_client_module.iter_rows
META = {op.operation_id: op for op in OPERATIONS}
SUMMARY = {}

def as_table(result, rows=PREVIEW_ROWS):
    if isinstance(result, list):
        return pd.DataFrame(result[:rows])
    if isinstance(result, dict) and result.get('type') == 'service_order':
        return Markdown('```json\n' + json.dumps(result, indent=1)[:4000] + '\n```')
    return Markdown('```json\n' + json.dumps(result, indent=1, default=str)[:3000] + '\n```')

def run(operation_id, rows=PREVIEW_ROWS):
    """Run one operation once (setup, execute, validate, teardown) and show its result.
    A failure is recorded in SUMMARY instead of stopping the notebook."""
    scenario, meta = SCENARIOS[operation_id], META[operation_id]
    result, elapsed, passed, error = None, None, False, None
    calls, couch_rows = [], [0]
    def counting(self, *args, **kwargs):  # counts the HTTP requests of execute() only
        calls.append(args[:2])
        return ORIGINAL_REQUEST(self, *args, **kwargs)
    def counting_json(self, *args, **kwargs):  # rows/documents CouchDB sends back
        answer = ORIGINAL_JSON(self, *args, **kwargs)
        if isinstance(answer, dict):
            couch_rows[0] += len(answer.get('rows') or answer.get('docs') or []) or ('_id' in answer)
        return answer
    def counting_get_doc(self, *args, **kwargs):  # single documents read by id
        answer = ORIGINAL_GET_DOC(self, *args, **kwargs)
        couch_rows[0] += answer is not None
        return answer
    def counting_rows(chunks):
        for row in ORIGINAL_ITER_ROWS(chunks):
            couch_rows[0] += 1
            yield row
    try:
        ctx = scenario.setup()
        try:
            CouchClient.request, CouchClient.json = counting, counting_json
            CouchClient.get_doc = counting_get_doc
            couch_client_module.iter_rows = counting_rows
            try:
                started = time.perf_counter()
                result = scenario.execute(ctx)
                elapsed = (time.perf_counter() - started) * 1000
            finally:
                CouchClient.request, CouchClient.json = ORIGINAL_REQUEST, ORIGINAL_JSON
                CouchClient.get_doc = ORIGINAL_GET_DOC
                couch_client_module.iter_rows = ORIGINAL_ITER_ROWS
            passed = bool(scenario.validate(ctx, result))
        finally:
            scenario.teardown(ctx)
    except Exception as exc:
        error = f'{type(exc).__name__}: {exc}'[:500]
    count = len(result) if isinstance(result, list) else None
    SUMMARY[operation_id] = {'operation_id': operation_id, 'part2': meta.part2_query,
                             'kind': meta.kind, 'description': meta.description,
                             'requirements': ', '.join(meta.requirement_ids), 'rows': count,
                             'requests': len(calls), 'couch_rows': couch_rows[0],
                             'time_ms': round(elapsed, 1) if elapsed is not None else None,
                             'correct': passed, 'error': error}
    head = f'**{operation_id}**' + (f' (Part 2 {meta.part2_query})' if meta.part2_query else '')
    if error:
        display(Markdown(f'{head}: {meta.description}. **ERROR** `{error}`'))
        return None
    display(Markdown(f'{head}: {meta.description}. '
                     + (f'{count:,} rows, ' if count is not None else '')
                     + f'{len(calls)} HTTP requests, {couch_rows[0]:,} rows/documents from CouchDB, '
                     + f'{elapsed:.1f} ms, validation **{"PASSED" if passed else "FAILED"}**'))
    return as_table(result, rows)

### PF-01: Orders and trips (create, read)

In [ ]:
run('PF-01-Q01')

In [ ]:
run('PF-01-Q02')

In [ ]:
run('PF-01-Q03')

### PF-02: Driver assignments

In [ ]:
run('PF-02-Q01')

In [ ]:
run('PF-02-Q02')

In [ ]:
run('PF-02-Q03')

In [ ]:
run('PF-02-Q04')

### PF-03: Service status

In [ ]:
run('PF-03-Q01')

In [ ]:
run('PF-03-Q02')

### PF-04: Payments

In [ ]:
run('PF-04-Q01')

In [ ]:
run('PF-04-Q02')

In [ ]:
run('PF-04-Q03')

### PF-05: Outlet search by area

In [ ]:
run('PF-05-Q01')

### PF-06: Service areas and outlets

In [ ]:
run('PF-06-Q01')

In [ ]:
run('PF-06-Q02')

### PF-07: Deletion

In [ ]:
run('PF-07-Q01')

In [ ]:
run('PF-07-Q02')

### PF-08: Driver and vehicle performance

In [ ]:
run('PF-08-Q01')

In [ ]:
run('PF-08-Q02')

In [ ]:
run('PF-08-Q03')

### PF-09: Activity and demand

In [ ]:
run('PF-09-Q01')

In [ ]:
run('PF-09-Q02')

### PF-10: Payments and revenue

In [ ]:
run('PF-10-Q01')

In [ ]:
run('PF-10-Q02')

In [ ]:
run('PF-10-Q03')

### Summary

In [ ]:
summary = pd.DataFrame(SUMMARY.values()).sort_values('operation_id').reset_index(drop=True)
summary.to_csv(OUT_DIR / 'operations_once.csv', index=False)
print(f"{summary['correct'].sum()}/{len(summary)} operations returned the expected result")
summary

## 5. Query the database yourself

The client wraps the CouchDB HTTP API one request per call, so any request can be tried here.
`DB` holds the database names, e.g. `DB['orders'] == 'grab_orders'`.

In [ ]:
# One order document: subtype, payment, trips and assignments together.
first_order = admin.all_docs(DB['orders'], startkey='ORD', endkey='ORD\ufff0', limit=1)['rows'][0]['id']
admin.get_doc(DB['orders'], first_order)

In [ ]:
# The design documents and their views (section 2.4.1).
pd.DataFrame([{'database': settings.db(name), 'design_doc': ddoc['_id'], 'view': view,
               'reduce': spec.get('reduce', '')}
              for name in DATABASES for ddoc in design.view_ddocs(name)
              for view, spec in ddoc['views'].items()])

In [ ]:
# A view query: trips per service area (grouped _count), like R5 before joining area names.
pd.DataFrame(admin.view(DB['orders'], 'analytics', 'trips_by_start_area', group=True)['rows'])

In [ ]:
# A Mango query and how CouchDB executes it: the index it uses and how much it reads.
query = {
    'selector': {'location.service_area_id': 'SAR0000007', 'outlet_status': 'OPEN'},
    'sort': [{'location.service_area_id': 'asc'}, {'outlet_status': 'asc'},
             {'outlet_name': 'asc'}, {'merchant_outlet_id': 'asc'}],
    'fields': ['merchant_outlet_id', 'outlet_name'],
    'use_index': ['mango-outlets', 'open-outlets-by-area'],
    'limit': 5, 'execution_stats': True,
}
found = admin.find(DB['geo'], query)
explain = admin.json('POST', f"/{DB['geo']}/_explain", body=query)
print('index used:', explain['index']['ddoc'], explain['index']['name'])
print('execution stats:', found.get('execution_stats'))
pd.DataFrame(found['docs'])

In [ ]:
# Write your own query here, e.g.:
# admin.view(DB['orders'], 'ops', 'by_customer', startkey=['CUS0000001'], endkey=['CUS0000001', {}])

## 6. Benchmark (optional)

Runs `benchmarks/run.py` with the CouchDB adapter: one warm-up and three measured runs per
operation, timing only `execute()`. Results go to `benchmarks/results/` (`runs.csv`,
`results.json`); copy them to `benchmarks/published/` once they are final.

In [ ]:
if RUN_BENCHMARK:
    from benchmarks import run as runner
    dataset_size = json.loads((DATA_DIR / 'manifest.json').read_text())['tables']['service_orders']['rows'] \
        if (DATA_DIR / 'manifest.json').is_file() else sum(1 for _ in open(DATA_DIR / 'service_orders.csv')) - 1
    runner.main(['--adapter', 'benchmarks.adapters.couchdb', '--dataset-size', str(dataset_size),
                 '--output-dir', str(OUT_DIR)])

results_file = OUT_DIR / 'results.json'
if results_file.is_file():
    results = json.loads(results_file.read_text())['results']
    table = pd.DataFrame([{
        'operation_id': r['operation_id'], 'part2': META[r['operation_id']].part2_query,
        'scenario': r['scenario'], 'correct': r['validation']['passed'],
        'runs_ms': r['measurements_ms'], 'mean_ms': r['mean_ms'], 'median_ms': r['median_ms'],
    } for r in results if 'operation_id' in r and r.get('dbms') == 'couchdb'])
    table = table.sort_values('operation_id').reset_index(drop=True)
    table.to_csv(OUT_DIR / 'benchmark_summary.csv', index=False)
    print(f"{int(table['correct'].sum())}/{len(table)} scenarios passed in every measured run")
    display(table)
else:
    print('No benchmark results yet: set RUN_BENCHMARK = True.')

## 7. Integrity-rule tests

The tests write valid and invalid documents into temporary databases and check that CouchDB
accepts or rejects each one (Part 2 constraints, status transitions, specialization, ID
immutability, revision conflicts). Their output is saved with the other evidence.

In [ ]:
if RUN_TESTS:
    tests = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_couchdb_rules.py',
                            'tests/test_couchdb_units.py', '-v', '-p', 'no:cacheprovider'],
                           capture_output=True, text=True, cwd=ROOT)
    (OUT_DIR / 'tests.txt').write_text(tests.stdout + tests.stderr)
    print(tests.stdout[-3000:])

## 8. Analisis hasil (bahan Bab 3.1–3.3)

Bagian ini membaca bukti di `OUT_DIR` dan menyajikannya sebagai grafik dan tabel. Setiap grafik
juga disimpan sebagai PNG di `OUT_DIR/figures/` (ikut dicatat di `MANIFEST.json`), dan selalu
didampingi tabel datanya.

Kriteria waktu mengikuti **Tabel 3.9 laporan Part 2**: kueri operasional ≤ 100 ms,
transaksional ≤ 200 ms, analitik ≤ 2 detik. Sebuah operasi **lulus** bila run *paling lambat*-nya
masih di bawah target, sama seperti di Part 2.

| Kategori | Operasi CouchDB |
|---|---|
| Operasional (≤ 100 ms) | PF-01-Q01, PF-01-Q02, PF-03-Q01, PF-05-Q01 (R10) |
| Transaksional (≤ 200 ms) | W1–W10 dan penghapusan (PF-07) |
| Analitik (≤ 2 s) | R1–R9 |

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter

FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'
GRID, AXIS, SURFACE = '#e1e0d9', '#c3c2b7', '#fcfcfb'
SERIES = ['#2a78d6', '#eb6834', '#1baf7a']   # categorical slots 1-3 (validated, all pairs)
LIGHT_BLUE, DARK_BLUE = '#86b6ef', '#1c5cab'  # one hue, two shades (dumbbell)
GOOD, CRITICAL = '#0ca30c', '#d03b3b'         # status, always paired with a ✓ / ✗ label
CATEGORY = {'read': 'Operasional', 'create': 'Transaksional', 'update': 'Transaksional',
            'delete': 'Transaksional', 'analytics': 'Analitik'}
TARGET_MS = {'Operasional': 100, 'Transaksional': 200, 'Analitik': 2000}
CATEGORY_COLOR = dict(zip(TARGET_MS, SERIES))

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.edgecolor': AXIS, 'axes.labelcolor': INK2, 'xtick.color': MUTED, 'ytick.color': INK2,
    'axes.spines.top': False, 'axes.spines.right': False, 'grid.color': GRID,
    'grid.linewidth': 0.6, 'legend.frameon': False, 'lines.linewidth': 2,
})

def finish(fig, name):
    """Save the figure for the report and the audit trail, then show it."""
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{name}.png', dpi=200, bbox_inches='tight')
    plt.show()

def plain_log(ax, axis='x'):
    """Log axis with plain tick labels (20, 50, 100, 1,000) instead of powers of ten."""
    for name in ('x', 'y') if axis == 'both' else (axis,):
        scale = ax.xaxis if name == 'x' else ax.yaxis
        low, high = ax.get_xlim() if name == 'x' else ax.get_ylim()
        decades = abs(np.log10(max(high, 1e-9) / max(low, 1e-9)))
        scale.set_major_locator(LogLocator(base=10, subs=(1,) if decades > 4 else (1, 2, 5)))
        scale.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}' if v >= 1 else f'{v:g}'))
        scale.set_minor_formatter(NullFormatter())

def gridlines(ax, axis='x'):
    ax.grid(axis=axis)
    ax.set_axisbelow(True)

def read_evidence(name):
    path = OUT_DIR / name
    return json.loads(path.read_text()) if path.is_file() else None

load_info = read_evidence('load_report.json')
verify_info = read_evidence('verify_report.json')
bench = read_evidence('results.json')
bench_rows = [r for r in (bench or {}).get('results', []) if r.get('dbms') == 'couchdb']
once = (pd.DataFrame(SUMMARY.values()) if SUMMARY
        else pd.read_csv(OUT_DIR / 'operations_once.csv') if (OUT_DIR / 'operations_once.csv').is_file()
        else pd.DataFrame())
print('bukti tersedia:', {'load': bool(load_info), 'verifikasi': bool(verify_info),
                          'benchmark': len(bench_rows), 'eksekusi tunggal': len(once)})

### 8.1 Pemuatan data dan pembangunan indeks (Bab 3.1)

In [ ]:
if load_info:
    loads = pd.DataFrame([{'database': db, 'dokumen': v['documents_written'],
                           'gagal': v['documents_failed'], 'detik': v['load_seconds'],
                           'dokumen_per_detik': v['docs_per_second']}
                          for db, v in load_info['databases'].items()])
    builds = pd.DataFrame(load_info.get('index_builds', []))
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
    ax = axes[0]
    ax.barh(loads['database'], loads['detik'], color=SERIES[0], height=0.55)
    for y, (seconds, rate) in enumerate(zip(loads['detik'], loads['dokumen_per_detik'])):
        ax.text(seconds, y, f'  {seconds:,.1f} s · {rate:,.0f} dok/s', va='center', color=INK2, fontsize=9)
    ax.invert_yaxis(); gridlines(ax); ax.margins(x=0.45)
    ax.set_xlabel('detik'); ax.set_title('Waktu pemuatan per database')
    ax = axes[1]
    if not builds.empty:
        labels = builds['database'] + '  ' + builds['design_doc'].str.replace('_design/', '', regex=False)
        ax.barh(labels, builds['build_seconds'], color=SERIES[0], height=0.55)
        for y, seconds in enumerate(builds['build_seconds']):
            ax.text(seconds, y, f'  {seconds:,.1f} s', va='center', color=INK2, fontsize=9)
        ax.invert_yaxis(); gridlines(ax); ax.margins(x=0.3)
    ax.set_xlabel('detik'); ax.set_title('Waktu build indeks per design document')
    finish(fig, '01_pemuatan_dan_indeks')
    display(loads)
    display(builds)
    print(f"Total pemuatan: {load_info['load_seconds_total']:,.1f} s · total build indeks: "
          f"{load_info.get('index_build_seconds_total') or 0:,.1f} s · CouchDB "
          f"{load_info['couchdb_version']}, batch {load_info['batch_size']}, {load_info['workers']} worker")
else:
    print('Belum ada load_report.json untuk dataset ini (jalankan dengan RUN_LOAD = True).')

### 8.2 Kesetaraan data dengan model relasional (Bab 3.1)

Lingkaran abu-abu adalah jumlah baris di CSV (data relasional Part 2), titik biru adalah jumlah
baris yang tersimpan di CouchDB. Titik tepat di tengah lingkaran berarti jumlahnya sama.

In [ ]:
if verify_info:
    tables = pd.DataFrame(verify_info['tables'])
    y = list(range(len(tables)))
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.scatter(tables['csv_rows'], y, s=150, facecolor='none', edgecolor=MUTED, linewidth=1.5,
               label='baris di CSV (relasional)', zorder=2)
    ax.scatter(tables['couchdb_rows'], y, s=40, color=SERIES[0], label='baris di CouchDB', zorder=3)
    ax.set_yticks(y, [f"{'✓' if m else '✗'}  {t}  ({d.replace('grab_', '')})"
                      for t, m, d in zip(tables['table'], tables['match'], tables['database'])])
    for yy, rows_ in zip(y, tables['csv_rows']):
        ax.text(rows_ * 1.5, yy, f'{rows_:,}', va='center', color=INK2, fontsize=8.5)
    ax.set_xscale('log'); ax.set_xlim(right=tables['csv_rows'].max() * 12)
    plain_log(ax); ax.invert_yaxis(); gridlines(ax)
    ax.set_xlabel('jumlah baris (skala log)')
    ax.set_title(f"Jumlah baris per tabel Part 2: {int(tables['match'].sum())} dari {len(tables)} sama persis",
                 pad=26)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, fontsize=9)
    finish(fig, '02_kesetaraan_jumlah_baris')
    display(tables)
    samples = pd.DataFrame([{'entitas': k, 'dokumen_disampel': v['sampled'],
                             'tidak_cocok': len(v['mismatches'])}
                            for k, v in verify_info['samples'].items()])
    display(Markdown('**Sampel acak dokumen yang dibandingkan field demi field dengan baris CSV:**'))
    display(samples)
    print('documents per database:', verify_info['documents_per_database'])
else:
    print('Belum ada verify_report.json untuk dataset ini.')

### 8.3 Penyimpanan

In [ ]:
if load_info and load_info.get('database_sizes'):
    views = load_info.get('view_index_sizes') or {}
    storage = pd.DataFrame([{
        'database': db, 'dokumen': v['doc_count'],
        'data_dokumen_MB': ((v['sizes'] or {}).get('external') or 0) / 2**20,
        'file_database_MB': ((v['sizes'] or {}).get('file') or 0) / 2**20,
        'indeks_view_MB': sum(((s or {}).get('file') or 0) for k, s in views.items()
                              if k.startswith(db + '/')) / 2**20,
    } for db, v in load_info['database_sizes'].items()])
    measures = [('data_dokumen_MB', 'data dokumen (JSON)'), ('file_database_MB', 'file database'),
                ('indeks_view_MB', 'file indeks view')]
if load_info and load_info.get('database_sizes') and storage[[m for m, _ in measures]].to_numpy().sum() > 0:
    fig, ax = plt.subplots(figsize=(10, 3.8))
    height = 0.26
    for i, ((column, label), color) in enumerate(zip(measures, SERIES)):
        positions = [k + (i - 1) * (height + 0.02) for k in range(len(storage))]
        ax.barh(positions, storage[column], height=height, color=color, label=label)
    ax.set_yticks(range(len(storage)), storage['database'])
    ax.invert_yaxis(); gridlines(ax)
    ax.set_xlabel('MB'); ax.set_title('Ukuran penyimpanan per database', pad=26)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=3, fontsize=9)
    finish(fig, '03_penyimpanan')
if load_info and load_info.get('database_sizes'):
    storage.loc['total'] = storage.sum(numeric_only=True)
    storage.loc['total', 'database'] = 'total'
    display(storage.round(1))
else:
    print('Ukuran penyimpanan tidak tersedia di load_report.json.')

### 8.4 Kebenaran hasil dan waktu eksekusi terhadap target (Bab 3.2)

Titik abu-abu kecil adalah setiap run terukur, titik besar adalah rata-ratanya, garis tegak adalah
target. ✓ berarti run paling lambat masih di bawah target; ✗ berarti melewati target.

In [ ]:
if bench_rows:
    def slowest(runs):
        runs = [x for x in runs if x is not None]
        return max(runs) if runs else None
    timing = pd.DataFrame([{
        'operation_id': r['operation_id'], 'part2': META[r['operation_id']].part2_query or '',
        'operasi': META[r['operation_id']].description,
        'kategori': CATEGORY[META[r['operation_id']].kind],
        'run_ms': r['measurements_ms'], 'rata2_ms': r['mean_ms'], 'median_ms': r['median_ms'],
        'terlambat_ms': slowest(r['measurements_ms']),
        'hasil_benar': r['validation']['passed'],
    } for r in bench_rows])
    for column in ('rata2_ms', 'median_ms', 'terlambat_ms'):
        timing[column] = pd.to_numeric(timing[column], errors='coerce')
    timing['target_ms'] = timing['kategori'].map(TARGET_MS)
    timing['rasio_thd_target'] = (timing['terlambat_ms'] / timing['target_ms']).round(2)
    timing['lulus_target'] = timing['terlambat_ms'] <= timing['target_ms']

    counts = timing['kategori'].value_counts()
    fig, axes = plt.subplots(3, 1, figsize=(11, 2.2 + 0.42 * len(timing)),
                             gridspec_kw={'height_ratios': [counts.get(c, 1) + 1.5 for c in TARGET_MS]})
    for ax, (category, target) in zip(axes, TARGET_MS.items()):
        part = timing[timing['kategori'] == category].sort_values('rata2_ms', ascending=False)
        for i, (_, row) in enumerate(part.iterrows()):
            runs = [x for x in row['run_ms'] if x is not None]
            ax.scatter(runs, [i] * len(runs), s=16, color=MUTED, zorder=2)
            if pd.notna(row['rata2_ms']):
                ax.scatter([row['rata2_ms']], [i], s=70, zorder=3, edgecolor=SURFACE, linewidth=1.5,
                           color=GOOD if row['lulus_target'] else CRITICAL)
        ax.axvline(target, color=INK2, linewidth=1.2, zorder=1)
        ax.text(target, -0.9, f' target ≤ {target:,} ms', color=INK2, fontsize=9, va='center')
        ax.set_yticks(range(len(part)), [
            f"{'✓' if ok and good else '✗'}  {oid}" + (f' ({p})' if p else '')
            for oid, p, ok, good in zip(part['operation_id'], part['part2'],
                                        part['lulus_target'], part['hasil_benar'])])
        ax.set_ylim(len(part) - 0.5, -1.4)
        ax.set_xscale('log'); plain_log(ax); gridlines(ax)
        passed = int(part['lulus_target'].sum())
        ax.set_title(f'{category}: {passed} dari {len(part)} operasi dalam target ≤ {target:,} ms')
    axes[-1].set_xlabel('waktu eksekusi (ms, skala log)')
    fig.legend(handles=[
        Line2D([], [], marker='o', linestyle='', color=MUTED, markersize=5, label='satu run terukur'),
        Line2D([], [], marker='o', linestyle='', color=GOOD, markersize=8,
               label='rata-rata; ✓ semua run ≤ target'),
        Line2D([], [], marker='o', linestyle='', color=CRITICAL, markersize=8,
               label='rata-rata; ✗ ada run > target'),
        Line2D([], [], color=INK2, linewidth=1.2, label='target (Tabel 3.9 Part 2)'),
    ], loc='upper right', ncol=4, fontsize=9, bbox_to_anchor=(1, 1.01))
    finish(fig, '04_waktu_eksekusi_vs_target')
    shown = timing.drop(columns=['run_ms']).sort_values('operation_id').reset_index(drop=True)
    shown.to_csv(OUT_DIR / 'analysis_timing.csv', index=False)
    display(Markdown(
        f"**Hasil benar:** {int(timing['hasil_benar'].sum())} dari {len(timing)} operasi benar di "
        f"setiap run terukur. **Dalam target:** {int(timing['lulus_target'].sum())} dari {len(timing)}."))
    display(shown)
else:
    print('Belum ada hasil benchmark untuk dataset ini (jalankan dengan RUN_BENCHMARK = True).')

### 8.5 Faktor yang memengaruhi waktu eksekusi

Dua faktor yang diukur langsung dari eksekusi tunggal di bagian 4: **jumlah baris/dokumen yang
dikirim CouchDB ke client** (untuk kueri top-N dan agregasi, CouchDB mengirim semua grup view
lalu client memilih atau menggabungkannya, jadi angka ini bisa jauh lebih besar dari hasil
akhir), dan **jumlah request HTTP** per operasi (CouchDB tidak punya join atau transaksi
multi-dokumen, jadi keduanya dikerjakan dengan beberapa request).

In [ ]:
if bench_rows and not once.empty and timing['rata2_ms'].notna().any():
    has_requests = 'requests' in once.columns
    has_couch_rows = 'couch_rows' in once.columns
    factors = timing.merge(once[['operation_id', 'rows'] + (['requests'] if has_requests else [])
                                + (['couch_rows'] if has_couch_rows else [])],
                           on='operation_id', how='left')
    measure = 'couch_rows' if has_couch_rows else 'rows'
    factors['baris_hasil'] = factors[measure].fillna(1).clip(lower=1)
    if has_requests:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={'width_ratios': [1.25, 1]})
    else:  # evidence from a run that did not count HTTP requests yet
        fig, axis = plt.subplots(figsize=(9, 5.2))
        axes = [axis]
    ax = axes[0]
    for category, color in CATEGORY_COLOR.items():
        part = factors[factors['kategori'] == category]
        ax.scatter(part['baris_hasil'], part['rata2_ms'], s=60, color=color, edgecolor=SURFACE,
                   linewidth=1.5, label=category, zorder=3)
    for _, row in factors.nlargest(5, 'rata2_ms').iterrows():
        ax.annotate(row['operation_id'] + (f" ({row['part2']})" if row['part2'] else ''),
                    (row['baris_hasil'], row['rata2_ms']), textcoords='offset points',
                    xytext=(7, 3), fontsize=8.5, color=INK2)
    ax.set_xscale('log'); ax.set_yscale('log'); plain_log(ax, 'both'); gridlines(ax, 'both')
    ax.set_xlabel(('baris/dokumen yang dikirim CouchDB ke client' if has_couch_rows
                   else 'baris hasil akhir (operasi tulis dihitung 1)') + ' (skala log)')
    ax.set_ylabel('rata-rata waktu eksekusi (ms, skala log)')
    ax.set_title('Waktu eksekusi vs jumlah baris' + (' yang dikirim CouchDB' if has_couch_rows
                                                       else ' hasil'), pad=26)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=3, fontsize=9)
    if has_requests:
        ax = axes[1]
        order = factors.sort_values(['requests', 'operation_id'], ascending=[False, True])
        ax.barh(order['operation_id'] + order['part2'].map(lambda p: f' ({p})' if p else ''),
                order['requests'], color=[CATEGORY_COLOR[c] for c in order['kategori']], height=0.6)
        for y, n in enumerate(order['requests']):
            ax.text(n, y, f' {int(n)}', va='center', color=INK2, fontsize=8.5)
        ax.invert_yaxis(); gridlines(ax); ax.tick_params(axis='y', labelsize=8.5)
        ax.set_xlabel('request HTTP per operasi'); ax.set_title('Jumlah request HTTP per operasi', pad=26)
        ax.legend(handles=[Line2D([], [], marker='s', linestyle='', color=c, markersize=8, label=k)
                           for k, c in CATEGORY_COLOR.items()], loc='lower left', bbox_to_anchor=(0, 1.0),
                  ncol=3, fontsize=9)
    finish(fig, '05_faktor_baris_dan_request')
    columns = ['operation_id', 'part2', 'kategori', 'rows'] + (['couch_rows'] if has_couch_rows else []) \
        + (['requests'] if has_requests else []) + ['rata2_ms', 'target_ms', 'lulus_target']
    display(factors[columns].sort_values('rata2_ms', ascending=False).reset_index(drop=True))
    if not has_requests:
        print('Jumlah request HTTP tidak tercatat di operations_once.csv run ini (fitur ditambahkan kemudian).')

### 8.6 Pengaruh volume data: sampel 40 ribu vs data penuh 2 juta order

Muncul bila kedua dataset sudah dijalankan dengan `RUN_BENCHMARK = True` (folder
`couchdb_sample` dan `couchdb_canonical`).

In [ ]:
runs_by_dataset = {}
for folder in sorted((ROOT / 'benchmarks' / 'results').glob('couchdb_*')):
    if (folder / 'results.json').is_file():
        doc = json.loads((folder / 'results.json').read_text())
        runs_by_dataset[folder.name.removeprefix('couchdb_')] = {
            r['operation_id']: r['mean_ms'] for r in doc['results'] if r.get('dbms') == 'couchdb'}
small, full = runs_by_dataset.get('sample', {}), runs_by_dataset.get('canonical', {})
scale = pd.DataFrame([{'operation_id': k, 'part2': META[k].part2_query or '',
                       'sampel_40rb_ms': small[k], 'penuh_2jt_ms': full[k]}
                      for k in small if small.get(k) and full.get(k)])
if len(scale):
    scale['kenaikan_kali'] = (scale['penuh_2jt_ms'] / scale['sampel_40rb_ms']).round(1)
    scale = scale.sort_values('penuh_2jt_ms', ascending=False).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(10, 0.6 + 0.34 * len(scale)))
    for y, row in scale.iterrows():
        ax.plot([row['sampel_40rb_ms'], row['penuh_2jt_ms']], [y, y], color=GRID, linewidth=2, zorder=1)
    ax.scatter(scale['sampel_40rb_ms'], scale.index, s=46, color=LIGHT_BLUE, zorder=2,
               label='sampel 40 ribu order')
    ax.scatter(scale['penuh_2jt_ms'], scale.index, s=46, color=DARK_BLUE, zorder=3,
               label='data penuh 2 juta order')
    ax.set_yticks(scale.index, scale['operation_id'] + scale['part2'].map(lambda p: f' ({p})' if p else ''))
    ax.invert_yaxis(); ax.set_xscale('log'); plain_log(ax); gridlines(ax)
    ax.tick_params(axis='y', labelsize=8.5)
    ax.set_xlabel('rata-rata waktu eksekusi (ms, skala log)')
    ax.set_title('Waktu eksekusi pada dua ukuran data', pad=26)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, fontsize=9)
    finish(fig, '06_pengaruh_volume_data')
    display(scale)
else:
    print('Perbandingan volume butuh hasil benchmark yang berhasil untuk kedua dataset '
          '(couchdb_sample dan couchdb_canonical); tersedia:', sorted(runs_by_dataset))

### 8.7 Pemenuhan kebutuhan Bab 1 dan ringkasan temuan

In [ ]:
tests_text = (OUT_DIR / 'tests.txt').read_text() if (OUT_DIR / 'tests.txt').is_file() else ''
passed_tests = re.search(r'(\d+) passed', tests_text)
failed_tests = re.search(r'(\d+) failed', tests_text)
if bench_rows:
    requirement_rows = []
    for requirement in sorted({r for op in OPERATIONS for r in op.requirement_ids if r.startswith('PF')}):
        ops = [op.operation_id for op in OPERATIONS if requirement in op.requirement_ids]
        part = timing[timing['operation_id'].isin(ops)]
        requirement_rows.append({
            'kebutuhan': requirement, 'operasi': ', '.join(ops),
            'semua_hasil_benar': '✓' if part['hasil_benar'].all() else '✗',
            'semua_dalam_target': '✓' if part['lulus_target'].all() else '✗ ' + ', '.join(
                part.loc[~part['lulus_target'], 'operation_id'])})
    display(pd.DataFrame(requirement_rows))
    rows_total = sum(t['csv_rows'] for t in verify_info['tables']) if verify_info else None
    docs_total = sum(verify_info['documents_per_database'].values()) if verify_info else None
    display(pd.DataFrame([
        {'kebutuhan': 'PNF-03 Kinerja akses data',
         'bukti': f"{int(timing['lulus_target'].sum())}/{len(timing)} operasi dalam target Tabel 3.9 Part 2"},
        {'kebutuhan': 'PNF-04 Konsistensi dan integritas',
         'bukti': f"tes aturan integritas: {passed_tests.group(1) if passed_tests else 0} lulus, "
                  f"{failed_tests.group(1) if failed_tests else 0} gagal (lihat tests.txt)"},
        {'kebutuhan': 'PNF-06 Fleksibilitas struktur data',
         'bukti': 'subtipe food/mobility/delivery tersimpan sebagai objek tertanam dengan atribut berbeda'},
        {'kebutuhan': 'PNF-07 Efisiensi sumber daya',
         'bukti': (f'{rows_total:,} baris relasional tersimpan dalam {docs_total:,} dokumen'
                   if rows_total else 'lihat 8.3')},
        {'kebutuhan': 'PNF-01, PNF-02, PNF-05',
         'bukti': 'tidak diuji: CouchDB berjalan sebagai satu node, tanpa replikasi atau cluster'},
    ]))
    ranked = timing.dropna(subset=['rata2_ms']).sort_values('rata2_ms')
    fastest = ', '.join(f"{r.operation_id} {r.rata2_ms:,.1f} ms" for r in ranked.head(3).itertuples())
    slowest_ = ', '.join(f"{r.operation_id} {r.rata2_ms:,.1f} ms" for r in ranked.tail(3)[::-1].itertuples())
    over = timing[~timing['lulus_target']].sort_values('rasio_thd_target', ascending=False)
    lines = [
        f"- Dataset `{DATASET}`: {environment['dataset_orders']:,} order, CouchDB {environment['couchdb_version']}, "
        f"commit `{environment['git_commit'][:7]}`.",
        f"- Hasil benar: {int(timing['hasil_benar'].sum())}/{len(timing)} operasi; dalam target: "
        f"{int(timing['lulus_target'].sum())}/{len(timing)}.",
        f"- Tercepat: {fastest}.",
        f"- Terlambat: {slowest_}.",
    ]
    if len(over):
        lines.append('- Melewati target: ' + ', '.join(
            f"{r.operation_id}{' (' + r.part2 + ')' if r.part2 else ''} {r.rasio_thd_target}× target"
            for r in over.itertuples()) + '.')
    if load_info:
        lines.append(f"- Pemuatan {load_info['load_seconds_total']:,.0f} s, build indeks "
                     f"{load_info.get('index_build_seconds_total') or 0:,.0f} s.")
    summary_md = '\n'.join(lines)
    (OUT_DIR / 'analysis_summary.md').write_text(summary_md + '\n')
    display(Markdown('**Ringkasan temuan (otomatis dari bukti):**\n\n' + summary_md))

## 9. Audit manifest

The SHA-256 of every evidence file (including the figures) is written to `MANIFEST.json`;
with `PUBLISH = True` the evidence is copied to `benchmarks/published/` so it can be committed.

In [ ]:
environment['run_finished_utc'] = datetime.now(UTC).isoformat()
(OUT_DIR / 'environment.json').write_text(json.dumps(environment, indent=2) + '\n')
files = {f.name: {'bytes': f.stat().st_size, 'sha256': sha256(f)}
         for f in sorted(OUT_DIR.iterdir()) if f.is_file() and f.name != 'MANIFEST.json'}
files[REFERENCE_FILE.name] = {'bytes': REFERENCE_FILE.stat().st_size, 'sha256': sha256(REFERENCE_FILE),
                              'note': 'expected results, kept in data/reference (not published)'}
(OUT_DIR / 'MANIFEST.json').write_text(json.dumps(files, indent=2) + '\n')
if PUBLISH:
    target = ROOT / 'benchmarks' / 'published' / OUT_DIR.name
    shutil.copytree(OUT_DIR, target, dirs_exist_ok=True)
    print(f'Evidence copied to {target.relative_to(ROOT)} (commit it to keep it)')
pd.DataFrame(files).T